# tensor-to-device — ex2: align a list of tensors to a common device

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-to-device`. Running the final beacon cell reports progress against the `PyTorch: tensor.to(device)` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: tensor.to(device)` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tensor-to-device`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-to-device"
DD_SUBTOPIC = "PyTorch: tensor.to(device)"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `tensor.to(device)` — quick refresher

`x.to(device)` is **not in-place** for tensors — it returns a (possibly new) tensor on the target device. You must capture the return value. Contrast with `model.to(device)`, which IS in-place on the Module's parameters/buffers.

**Device-mismatch errors.** Any binary op (add, matmul, ...) between a CPU tensor and a CUDA tensor raises `RuntimeError: Expected all tensors to be on the same device`. The fix is to move every input to one common device BEFORE the op. The idiomatic helper aligns a *list* of tensors in one pass.

**Idempotence.** `x.to(x.device)` returns `x` itself (same object). `x.to(other_device)` returns a fresh tensor. The `is` check exposes this.

### Exercise 2 — align a list of tensors to a common device

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `.to(device)` across a list of tensors so that all results live on the target device, exploiting `.to` idempotence so already-on-device tensors are returned as the SAME object.
> Keywords: device, alignment, to, idempotence
> ```

**KCs targeted:** `to-is-not-inplace`, `pick-device-with-cuda-available`

Implement `ex2_align_to_device(tensors, device)`. Given a list of tensors (each possibly on a different device, possibly the same dtype or different) and a target `device` string (e.g. `'cpu'`), return a list of the same length where every tensor lives on `device`.

**Hard requirements.**
1. Use `.to(device)` on each tensor — do not write a manual copy or `clone()` fallback.
2. **Idempotence**: if a tensor is ALREADY on `device`, `x.to(device)` must return the same object (`out is x` must be True). Don't add `clone()` after `.to()`.
3. **Preserve order**: `out[i]` corresponds to `tensors[i]`.
4. **No mutation**: do not modify the input list or its tensors.

Inputs:
- `tensors`: list of `Tensor`.
- `device`: str, one of `'cpu'` or `'cuda'` or `'cuda:0'` etc.

Output: list of `Tensor`, each on `device`.

In [ ]:
def ex2_align_to_device(tensors: list, device: str) -> list:
    """Move every tensor in `tensors` to `device`."""
    raise NotImplementedError()


def _test_ex2():
    # Single-device suite (CPU-only) — the contract still applies.
    a = t.arange(3)
    b = t.zeros(2, 4)
    c = t.tensor([1.5, 2.5])
    out = ex2_align_to_device([a, b, c], 'cpu')
    assert isinstance(out, list) and len(out) == 3
    for i, (orig, new) in enumerate(zip([a, b, c], out)):
        assert new.device.type == 'cpu', (
            f'out[{i}] on {new.device}, expected cpu'
        )
        # Idempotence: already-on-cpu tensors must return the same object.
        assert new is orig, (
            f'out[{i}] is a NEW object — did you accidentally clone()? '
            f'x.to(x.device) must return x itself.'
        )

    # Empty list — degenerate but must not crash.
    assert ex2_align_to_device([], 'cpu') == []

    # Original list is not mutated.
    src = [t.arange(4), t.ones(2)]
    src_copy = list(src)
    _ = ex2_align_to_device(src, 'cpu')
    assert src == src_copy, 'do not mutate the input list (rebinding allowed; reordering not)'
    assert len(src) == 2

    # Order preserved.
    named = [t.tensor([float(i)]) for i in range(5)]
    moved = ex2_align_to_device(named, 'cpu')
    for i, m in enumerate(moved):
        assert m.item() == float(i), f'order broken at {i}: got {m.item()}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_align_to_device(tensors, device):
    return [x.to(device) for x in tensors]
```

**One-liner — and that's the whole point.** PyTorch's `.to()` already gives you idempotence for free: `x.to(x.device)` returns `x` (the exact same Python object). A list comprehension is enough.

**Common over-engineering** — DON'T do this:
```python
return [x.to(device) if x.device != device else x.clone() for x in tensors]
```
The `clone()` allocates fresh storage you didn't ask for, breaking the idempotence contract and wasting memory in inner loops.

**When CUDA is available**, the same code moves tensors across the PCI-e bus. The idempotence guarantee still holds: any tensor already on `cuda:0` is returned as-is (no D2D copy).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()